Donations:
Business questions:
1. Which donors are likely to donate again?
2. Which campaigns generate higher lifetime value?
3. Does allocation transparency affect future donations?
4. Does non-monetary engagement lead to later donations?

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path relative to this notebook (ML pipelines/jupyter/ -> sibling lighthouse_csv_v7/)
_donations_csv = Path("..") / "lighthouse_csv_files" / "donations.csv"
_donation_allocations_csv = Path("..") / "lighthouse_csv_files" / "donation_allocations.csv"
donations = pd.read_csv(_donations_csv.resolve())
allocations = pd.read_csv(_donation_allocations_csv.resolve())
donations.head()

,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN
1,2,25,Time,2025-12-02,True,Year-End Hope,Event,NaN,NaN,35.15,hours,Community outreach support,NaN
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN
3,4,33,Monetary,2023-09-11,False,NaN,PartnerReferral,PHP,1230.56,1230.56,pesos,In support of safehouse operations,NaN
4,5,24,InKind,2023-11-08,True,GivingTuesday,SocialMedia,NaN,NaN,1177.41,items,In support of safehouse operations,421.0


In [11]:
allocations.head()

,allocation_id,donation_id,safehouse_id,program_area,amount_allocated,allocation_date,allocation_notes
0,1,1,2,Education,717.18,2025-12-31,NaN
1,2,2,4,Transport,35.15,2025-12-02,NaN
2,3,3,8,Wellbeing,1074.65,2024-12-02,NaN
3,4,4,9,Operations,799.86,2023-09-11,NaN
4,5,5,8,Operations,1177.41,2023-11-08,NaN


In [12]:
_by_area = (
    allocations.groupby(["donation_id", "program_area"], as_index=False)["amount_allocated"]
    .sum()
)
_wide = _by_area.pivot_table(
    index="donation_id",
    columns="program_area",
    values="amount_allocated",
    aggfunc="sum",
    fill_value=0,
)
_row_sum = _wide.sum(axis=1).replace(0, np.nan)
_share = _wide.div(_row_sum, axis=0).fillna(0)
_share.columns = [f"pct_{str(c).lower()}" for c in _share.columns]
alloc_features = _share.reset_index()

_summaries = allocations.groupby("donation_id").agg(
    num_allocations=("allocation_id", "count"),
    total_amount_allocated=("amount_allocated", "sum"),
).reset_index()

alloc_features = alloc_features.merge(
    _summaries,
    on="donation_id",
    how="left",
    validate="one_to_one",
)
alloc_features.head()

TypeError: Must provide 'func' or tuples of '(column, aggfunc).

In [ ]:
model_df = donations.merge(
    alloc_features,
    on="donation_id",
    how="left",
    validate="one_to_one",
)
model_df.head()

In [ ]:
assert len(model_df) == len(donations)
_pct_cols = [c for c in alloc_features.columns if c.startswith("pct_")]
assert np.allclose(alloc_features[_pct_cols].sum(axis=1), 1.0, rtol=1e-9, atol=1e-9)
_alloc_counts = allocations.groupby("donation_id").size().rename("n")
assert model_df.set_index("donation_id")["num_allocations"].equals(_alloc_counts)
"OK: one row per donation, allocation shares sum to 1, num_allocations matches raw allocations."